# Week 2 (v2) — Pull a *diverse* set of MdnC homologs & align them

**What changed from v1:** the first version pulled 80 near-identical copies of the same protein, so conservation looked uniform and uninformative. This version searches **wider** and then **samples across the family** — keeping a spread from close relatives (~95% identical) to distant ones (~30%). That diversity is what makes conservation meaningful next week.

**Milestone:** a FASTA of ~60–90 *diverse* MdnC relatives + an alignment where you can see genuinely conserved columns standing out against variable ones.

Run the cells top to bottom. The BLAST step is the slow one (10–20 min) — it's saved to disk, so you only wait once.


### Setup — imports and your email
NCBI requires an email so they can reach you if a script misbehaves.

In [1]:
from pathlib import Path
import time, shutil, statistics
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import Entrez, SeqIO

Entrez.email = "phamminh1@ufl.edu"     # <-- CHANGE THIS to your real email

assert "YOUR_EMAIL" not in Entrez.email, "Put your real email in the line above first."
print("Setup complete.")

Setup complete.


### Load your MdnC query from the 5IG9 file
Same as before — the longest chain in 5IG9 is MdnC, so we use that as the query.

In [2]:
DATA = Path("../data") if Path("../data").exists() else Path("data")
DATA.mkdir(exist_ok=True)

matches  = [p for p in DATA.glob("*.fasta") if "5ig9" in p.name.lower()]
matches += [p for p in DATA.glob("*.txt")   if "5ig9" in p.name.lower()]
if not matches:
    raise FileNotFoundError(f"Put the 5IG9 FASTA in {DATA.resolve()} and re-run this cell.")

records      = list(SeqIO.parse(matches[0], "fasta"))
query_record = max(records, key=lambda r: len(r.seq))
query_seq    = str(query_record.seq)
print(f"MdnC query: {len(query_seq)} residues | first 15: {query_seq[:15]}")

MdnC query: 333 residues | first 15: MTVLIVTFSRDNESI


In [3]:
print(query_seq[190:195])

EDLDN


### Step 1 — a WIDER BLAST (the slow step ☕☕)
We ask for up to 1000 hits so the search reaches the distant relatives, not just the near-identical pile. Saved to a new file so it runs fresh once.

In [4]:
blast_xml = DATA / "mdnC_blast_wide.xml"

if blast_xml.exists():
    print("Found a saved wide BLAST result - skipping the slow step.")
    print("(Delete", blast_xml.name, "to re-run from scratch.)")
else:
    print("Running a WIDER BLAST (up to 1000 hits)... this takes ~10-20 min. It's not frozen.")
    t0 = time.time()
    h = NCBIWWW.qblast("blastp", "nr", query_seq, hitlist_size=1000)
    blast_xml.write_text(h.read()); h.close()
    print(f"Done in {(time.time()-t0)/60:.1f} minutes. Saved to {blast_xml.name}")

Running a WIDER BLAST (up to 1000 hits)... this takes ~10-20 min. It's not frozen.
Done in 3.6 minutes. Saved to mdnC_blast_wide.xml


### Step 2 — parse hits and sample *across* the family
Instead of taking the top matches (all near-twins), we record each hit's **% identity to the query**, then bin by identity and take a capped number from each bin. That deliberately keeps close AND distant relatives — a real cross-section of the family.

In [5]:
# Filters: keep confident, well-covered hits (we drop the identity floor so distant relatives survive).
MAX_EVALUE   = 1e-5
MIN_COVERAGE = 60.0
QLEN = len(query_seq)

hits = []                                   # (accession, percent_identity_to_query)
with open(blast_xml) as fh:
    rec = NCBIXML.read(fh)
    for al in rec.alignments:
        b   = al.hsps[0]
        pid = 100.0 * b.identities  / b.align_length
        cov = 100.0 * b.align_length / QLEN
        if b.expect <= MAX_EVALUE and cov >= MIN_COVERAGE:
            hits.append((al.accession, pid))

# keep the highest identity seen per accession (removes duplicate accessions)
best = {}
for acc, pid in hits:
    best[acc] = max(pid, best.get(acc, 0))
hits = list(best.items())

if not hits:
    print("No hits passed. Try lowering MIN_COVERAGE, or your BLAST file may be empty (delete + re-run Step 1).")
else:
    lo, hi = min(p for _, p in hits), max(p for _, p in hits)
    print(f"{len(hits)} usable hits, identity to query ranges {lo:.0f}%-{hi:.0f}%")

# --- stratified sampling: spread across identity bins ---
BINS    = [(95, 100), (85, 95), (70, 85), (50, 70), (0, 50)]
PER_BIN = 18
chosen  = []
for lo, hi in BINS:
    inbin = [acc for acc, pid in hits if lo < pid <= hi]
    step  = max(1, len(inbin) // PER_BIN)         # walk through the bin so we sample it evenly
    picked = inbin[::step][:PER_BIN]
    chosen += picked
    print(f"  identity {lo:>3}-{hi:<3}%: {len(inbin):>4} available -> kept {len(picked)}")

chosen = list(dict.fromkeys(chosen))              # de-duplicate, keep order
print(f"\nSelected {len(chosen)} accessions spread across the family.")

996 usable hits, identity to query ranges 43%-100%
  identity  95-100%:  119 available -> kept 18
  identity  85-95 %:    2 available -> kept 2
  identity  70-85 %:  192 available -> kept 18
  identity  50-70 %:  474 available -> kept 18
  identity   0-50 %:  209 available -> kept 18

Selected 74 accessions spread across the family.


/opt/homebrew/Caskroom/miniforge/base/envs/microviridin/lib/python3.11/site-packages/Bio/Blast/NCBIXML.py:977: BiopythonParserWarning: NCBIXML: Ignored: '\nCREATE_VIEW\n\n\n'
  warnings.warn(


### Step 3 — download the full sequences and save the FASTA

In [6]:
out_fasta = DATA / "mdnC_homologs.fasta"
collected, seen_seqs = [], {query_seq}

for i in range(0, len(chosen), 50):
    handle = Entrez.efetch(db="protein", id=",".join(chosen[i:i+50]), rettype="fasta", retmode="text")
    for r in SeqIO.parse(handle, "fasta"):
        s = str(r.seq)
        if s and s not in seen_seqs:
            seen_seqs.add(s); collected.append(r)
    handle.close(); time.sleep(0.5)

query_record.id, query_record.description = "MdnC_5IG9_query", "MdnC query from 5IG9"
SeqIO.write([query_record] + collected, out_fasta, "fasta")
print(f"Saved {1 + len(collected)} sequences (query + {len(collected)} homologs) to {out_fasta.name}")

Saved 74 sequences (query + 73 homologs) to mdnC_homologs.fasta


### Step 4 — quick length sanity
Most sequences should be roughly MdnC-sized (~300–340 residues). The real diversity check comes right after the alignment, in Step 5.

In [7]:
seqs    = list(SeqIO.parse(out_fasta, "fasta"))
lengths = [len(r.seq) for r in seqs]
print(f"{len(seqs)} sequences | length -> min {min(lengths)} | median {int(statistics.median(lengths))} | max {max(lengths)}")
print("A pile of very short/long ones means junk slipped in - loosen or tighten filters in Step 2 if so.")

74 sequences | length -> min 320 | median 326 | max 343
A pile of very short/long ones means junk slipped in - loosen or tighten filters in Step 2 if so.


### Step 5 — align (auto-detects MAFFT or Clustal Omega) 🎯
Install once with `brew install mafft` if you haven't.

In [8]:
import subprocess
aln_out = DATA / "mdnC_homologs_aligned.fasta"

if shutil.which("mafft"):
    print("Aligning with MAFFT...")
    with open(aln_out, "w") as out:
        subprocess.run(["mafft", "--auto", str(out_fasta)], stdout=out, check=True)
elif shutil.which("clustalo"):
    print("Aligning with Clustal Omega...")
    subprocess.run(["clustalo", "-i", str(out_fasta), "-o", str(aln_out),
                    "--outfmt=fasta", "--force", "--threads=4"], check=True)
else:
    print("No aligner found. Run  brew install mafft  in Terminal, then re-run this cell.")
    aln_out = None

if aln_out and aln_out.exists():
    print("Alignment saved to", aln_out.name)
    # --- real diversity check: % identity to query, measured from the alignment ---
    aln   = list(SeqIO.parse(aln_out, "fasta"))
    q_rec = next((r for r in aln if "query" in r.id.lower()), aln[0])
    qs    = str(q_rec.seq)
    def id_to_q(s):
        same = tot = 0
        for a, b in zip(qs, s):
            if a != "-" and b != "-":
                tot += 1; same += (a == b)
        return 100.0 * same / tot if tot else 0
    ids = sorted(id_to_q(str(r.seq)) for r in aln if r is not q_rec)
    print(f"\nDiversity check -> % identity to query: min {min(ids):.0f} | median {statistics.median(ids):.0f} | max {max(ids):.0f}")
    if min(ids) < 90:
        print("Great - real diversity (distant relatives included). Conservation will be meaningful now.")
    else:
        print("Still all >90% identical - tell your helper and we'll seed the set with curated divergent sequences.")

Aligning with MAFFT...


outputhat23=16
treein = 0
compacttree = 0
stacksize: 8176 kb
rescale = 1
All-to-all alignment.
tbfast-pair (aa) Version 7.526
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a UPGMA tree ... 
   70 / 74
done.

Progressive alignment ... 
STEP    61 /73 
Reallocating..done. *alloclen = 1705
STEP    73 /73 
done.
tbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
1 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 16
sueff_global = 0.100000
nadd = 16
Loading 'hat3' ... done.
rescale = 1

   70 / 74
Segment   1/  1    1- 383
STEP 003-006-1  identical.   

Alignment saved to mdnC_homologs_aligned.fasta

Diversity check -> % identity to query: min 44 | median 70 | max 99
Great - real diversity (distant relatives included). Conservation will be meaningful now.


STEP 003-030-0  identical.   
Converged.

done
dvtditr (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strategy:
 L-INS-i (Probably most accurate, very slow)
 Iterative refinement method (<16) with LOCAL pairwise alignment information

If unsure which option to use, try 'mafft --auto input > output'.
For more information, see 'mafft --help', 'mafft --man' and the mafft page.

The default gap scoring scheme has been changed in version 7.110 (2013 Oct).
It tends to insert more gaps into gap-rich regions than previous versions.
To disable this change, add the --leavegappyregion option.



## ✅ Week 2 (v2) done

Files in `data/`: `mdnC_blast_wide.xml`, `mdnC_homologs.fasta`, `mdnC_homologs_aligned.fasta`.

**Commit to GitHub** with a message like *"Week 2 v2: diverse MdnC homologs + alignment."*

**Next (Week 3):** turn this alignment into a per-position conservation score, and map it onto the 5IG9 structure — now that the set is diverse, the truly important residues (your landmark set) should stand out from the background.
